In [1]:
import pandas as pd
from helpers import get_factor, get_price

In [2]:
CDF = pd.read_csv("../production/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

/tmp/ipykernel_2598570/3001582791.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))


In [3]:
smard = pd.read_csv("Gro_handelspreise_202301010000_202401010000_Stunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

/tmp/ipykernel_2598570/1081581146.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
/tmp/ipykernel_2598570/1081581146.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")


In [4]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [5]:
CDF2 = CDF.loc[CDF.produced_at > "2022-12-31 23:50"].loc[CDF.produced_at < "2024-01-01 00:00"]

In [6]:
dataset = CDF2.merge(seem, left_on="variable", right_on="sseid")

In [7]:
magic = dataset.groupby(["produced_at", "plantid"]).sum()

In [8]:
magic2 = magic[["value"]]

In [9]:
#magic2.sort_values(["produced_at", "value"])

In [10]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [11]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")

In [12]:
merged["revenue"] = merged["value"] * merged["price"]
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [13]:
tmp1.reset_index(inplace=True)

In [14]:
revenue = tmp1[["plantid", "revenue"]]

In [15]:
#merged

In [16]:
#dataset.sort_values(["plantid", "produced_at"])

In [17]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [18]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [19]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [20]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [21]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [37]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [38]:
tmp2 = pd.merge(tmp1, production, on="plantid")

In [40]:
tmp2 = tmp1

In [41]:
coal_cost_per_t = 103.5 or 120
co2_cost = 70
#electricity_price = 78.50

In [42]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [43]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [49]:
profit = tmp2[["plantid", "revenue", "profit"]]
profit["revenue"] = profit["revenue"].apply(lambda x: x / 10**6)

/tmp/ipykernel_2598570/4287809471.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  profit["revenue"] = profit["revenue"].apply(lambda x: x / 10**6)


In [52]:
profit.to_csv("profit.csv", index=False)

In [53]:
profit.sort_values("profit")

,plantid,revenue,profit
0,BB23020490,112.446805,-196.918413
12,BWpf-450-2948214-00000000,247.873328,-139.366872
33,NW500-0342658,118.681204,-106.696332
11,BWpf-450-2797933-00000000,266.390859,-99.171081
18,MV30000226,135.982097,-61.338277
9,BWpf-450-1741292-00000000,56.747003,-45.816059
20,NI01241117210,118.842134,-45.172941
3,BE166928,128.307113,-44.774484
15,BYS00114,16.650636,-42.992697
8,BWpf-450-1195689-00000000,19.508291,-31.911410
